# 04 — Cliff's Delta Effect SizeComputes Cliff's Delta as a non-parametric effect size measure on the 10-seed repeated-run results from `03_significance_testing.ipynb`, complementing the Wilcoxon p-values with an interpretable magnitude of separation between random-split and temporal-split performance. Adds the final column to Table 4.**Requires:** `significance_test_raw_runs.csv`, produced by `03_significance_testing.ipynb`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

SAVE_PATH = '/content/drive/MyDrive/New Approach/cicids2017-01/cicids2017/preprocessed'

# ── load your existing significance test results ──
df_runs = pd.read_csv(SAVE_PATH + '/significance_test_raw_runs.csv')
print(f"Loaded {len(df_runs)} rows")
print(df_runs.head())


def cliffs_delta(x, y):
    """
    Cliff's Delta — non-parametric effect size.
    Measures how often values in x exceed values in y, minus the
    reverse, normalized to [-1, 1].

    delta = (# x_i > y_j - # x_i < y_j) / (n_x * n_y)

    Interpretation thresholds (Romano et al. 2006):
      |delta| < 0.147           negligible
      0.147 <= |delta| < 0.33   small
      0.33  <= |delta| < 0.474  medium
      |delta| >= 0.474          large
    """
    x = np.asarray(x)
    y = np.asarray(y)
    n_x, n_y = len(x), len(y)

    # count pairwise comparisons
    greater = 0
    less = 0
    for xi in x:
        greater += np.sum(xi > y)
        less += np.sum(xi < y)

    delta = (greater - less) / (n_x * n_y)
    return delta


def interpret_delta(d):
    ad = abs(d)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    else:
        return "large"


# ══════════════════════════════════════════════════════════════
#  COMPUTE CLIFF'S DELTA FOR EVERY MODEL × METRIC PAIR
# ══════════════════════════════════════════════════════════════
models = df_runs['model'].unique()
metrics = ['acc', 'f1m', 'recall']
metric_labels = {'acc': 'Accuracy', 'f1m': 'F1-Macro', 'recall': 'Attack Recall'}

results = []
print('\n' + '='*70)
print("CLIFF'S DELTA EFFECT SIZE — RANDOM vs TEMPORAL")
print('='*70)

for model_name in models:
    sub = df_runs[df_runs.model == model_name]
    print(f'\n--- {model_name} (n={len(sub)} runs) ---')

    for metric in metrics:
        rand_col = f'{metric}_random'
        temp_col = f'{metric}_temporal'
        rand_vals = sub[rand_col].values
        temp_vals = sub[temp_col].values

        delta = cliffs_delta(rand_vals, temp_vals)
        interp = interpret_delta(delta)

        print(f'  {metric_labels[metric]:14s} | '
              f"Cliff's Delta = {delta:+.4f}  ({interp})")

        results.append({
            'model': model_name,
            'metric': metric_labels[metric],
            'cliffs_delta': round(delta, 4),
            'magnitude': interp,
        })

# ══════════════════════════════════════════════════════════════
#  SUMMARY TABLE — FOR PAPER
# ══════════════════════════════════════════════════════════════
delta_df = pd.DataFrame(results)
print('\n\n' + '='*70)
print("TABLE — EFFECT SIZE SUMMARY (FOR PAPER)")
print('='*70)
print(delta_df.to_string(index=False))

delta_df.to_csv(SAVE_PATH + '/cliffs_delta_summary.csv', index=False)
print(f"\nSaved → cliffs_delta_summary.csv")

# ══════════════════════════════════════════════════════════════
#  MERGE WITH EXISTING SIGNIFICANCE SUMMARY (if available)
# ══════════════════════════════════════════════════════════════
try:
    sig_summary = pd.read_csv(SAVE_PATH + '/significance_test_summary.csv')
    merged = sig_summary.merge(
        delta_df, on=['model', 'metric'], how='left'
    )
    print('\n\n' + '='*70)
    print("MERGED TABLE — Wilcoxon + Cliff's Delta (FOR PAPER TABLE 4)")
    print('='*70)
    print(merged[['model','metric','mean_inflation_pp','ci95_lo_pp',
                  'ci95_hi_pp','wilcoxon_pvalue','cliffs_delta',
                  'magnitude']].to_string(index=False))
    merged.to_csv(SAVE_PATH + '/significance_test_summary_with_effect_size.csv',
                 index=False)
    print(f"\nSaved → significance_test_summary_with_effect_size.csv")
except FileNotFoundError:
    print("\n(significance_test_summary.csv not found — "
          "run this after your significance testing script)")

Loaded 30 rows
          model  seed  acc_random  acc_temporal  f1m_random  f1m_temporal  \
0  RandomForest    42    0.993250      0.588916    0.989411      0.376306   
1      LightGBM    42    0.998425      0.589106    0.997514      0.371667   
2       XGBoost    42    0.998825      0.607756    0.998144      0.419067   
3  RandomForest    43    0.994650      0.614746    0.991585      0.440858   
4      LightGBM    43    0.998450      0.594736    0.997554      0.386548   

   recall_random  recall_temporal  
0       0.993775         0.006156  
1       0.998730         0.001022  
2       0.998603         0.046040  
3       0.993775         0.069449  
4       0.999238         0.014819  

CLIFF'S DELTA EFFECT SIZE — RANDOM vs TEMPORAL

--- RandomForest (n=10 runs) ---
  Accuracy       | Cliff's Delta = +1.0000  (large)
  F1-Macro       | Cliff's Delta = +1.0000  (large)
  Attack Recall  | Cliff's Delta = +1.0000  (large)

--- LightGBM (n=10 runs) ---
  Accuracy       | Cliff's Delta = +1.